In [ ]:
from collections import Counter
from pathlib import Path
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import random
from sympy import true

import part1 # to use code from tutorial
import torch

def set_seed(seed):
    """Seeds Python, NumPy (also used by sklearn's saga solver) and PyTorch (weight init, shuffling, dropout)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

# Shared Step: Create Datasets with Three Representations

In [ ]:
# splitting data into train and test sets and folds for cross-validation
LABEL_COL = "Party_orientation"
SAMPLE_FRAC = 1.0  # fraction of the data to use, 1.0 for all of it

# merge the party orientations into two classes, "Left" and "Right"
# "Centre" has very few samples, so it is discarded
orientation_to_side = {
    "Centre-left to left": "Left",
    "Centre-left": "Left",
    "Centre to centre-left": "Left",
    "Centre to centre-right": "Right",
    "Centre-right": "Right",
    "Right": "Right",
    "Right to far-right": "Right",
    "Far-right": "Right",
}

def merge_orientations(df, label_column=LABEL_COL):
    df = df[df[label_column].isin(orientation_to_side.keys())].copy()
    df[label_column] = df[label_column].map(orientation_to_side)
    return df

# get the dataframe: one dataframe with tokens, lemmas, sentences and embeddings,
# so all three representations are trained and evaluated on the same rows
df = merge_orientations(pd.read_pickle("./embeddings_sampled.pkl"))
df = df.sample(frac=SAMPLE_FRAC, random_state=42).reset_index(drop=True)

# train test split
train_df, test_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df[LABEL_COL])
train_df = train_df.reset_index(drop=True)

# labels and settings
label_names = sorted(df[LABEL_COL].unique())
outcome_space = part1.clslib.NamedClasses(label_names)
# repeated cross-validation: each seed gives its own fold split and model initialisation
# (fixed in advance; more runs give the Bayesian correlated t-test more paired differences to work with)
my_seeds = [0, 1, 2]
num_folds = 5

In [22]:
# to save results
results_folder = Path("./results")
results_folder.mkdir(exist_ok=True)
# to save models
my_classifiers = {}

In [ ]:
def get_vocab(texts, min_count=5, max_size=None):
    counts = Counter(item for text in texts for item in text.split())
    kept = [item for item, count in counts.most_common(max_size) if count >= min_count]
    return {item: i for i, item in enumerate(kept)}

class Encoder:
    def __len__(self) -> int:
        raise NotImplementedError

    def __call__(self, x: str) -> np.ndarray:
        raise NotImplementedError

def sparse_row(counts: dict, size: int) -> csr_matrix:
    """A 1 x size sparse row with counts[index] at each index (only the non-zero entries are stored)."""
    indices = sorted(counts)
    values = np.array([counts[i] for i in indices], dtype=np.float32)
    return csr_matrix((values, indices, [0, len(indices)]), shape=(1, size))

class TokenIndicator(Encoder):
    def __init__(self, vocab: dict):
        self.vocab = vocab

    def __len__(self) -> int:
        return len(self.vocab)

    def __call__(self, x: str) -> csr_matrix:
        # sparse row instead of np.zeros(len(vocab)), so the full dataset fits in memory
        indices = {self.vocab[item] for item in x.split() if item in self.vocab}
        return sparse_row({index: 1 for index in indices}, len(self))

# Function to create pairs of texts and labels so we can use functions from part1 (tutorial)
def make_pairs(df, column, label_names, label_column="Party_orientation"):
    label_to_id = {name: i for i, name in enumerate(label_names)}
    texts = df[column]
    # only join list columns (tokens/lemmas); str.join on a plain string column would put spaces between characters
    if isinstance(texts.iloc[0], (list, tuple, np.ndarray)):
        texts = texts.str.join(" ")
    labels = [label_to_id[name] for name in df[label_column]]
    return list(zip(texts, labels))


In [24]:
# cross-validation folds from the train rows only (the test set stays held out), using the tutorial's function
def make_cv(column):
    pairs = make_pairs(train_df, column, label_names)
    return [part1.experiment.cross_validation_split(pairs=pairs, k=num_folds, seed=seed) for seed in my_seeds]

cv_lemmas = make_cv("lemmas")        # for token indicator and bag of words
cv_sentences = make_cv("sentence")   # for the embeddings

In [ ]:
# evaluation: accuracy and macro F1, per cross-validation fold and on the held-out test set
# (macro F1 averages the F1 of Left and Right, so the smaller Left class counts as much as Right)
# scores are saved to results/<experiment>/cv_scores.csv and test_scores.csv, so they survive a kernel restart
from collections import OrderedDict
from sklearn.metrics import accuracy_score, f1_score

test_lemmas = make_pairs(test_df, "lemmas", label_names)
test_sentences = make_pairs(test_df, "sentence", label_names)

def add_cv_f1(name, data_splits):
    """Adds the macro F1 of every fold's validation predictions to the runs of experiment `name`, next to "accuracies",
    and saves accuracy / macro F1 per seed and fold to results/<name>/cv_scores.csv."""
    rows = []
    for seed, run, splits in zip(my_seeds, my_classifiers[name], data_splits):
        run["f1_scores"] = OrderedDict(
            (fold, f1_score([label for _, label in splits[fold]["validation"]], preds, average="macro"))
            for fold, preds in run["predictions"].items()
        )
        for fold in run["f1_scores"]:
            rows.append({"seed": seed, "fold": fold, "accuracy": run["accuracies"][fold], "f1": run["f1_scores"][fold]})
            print(f"seed {seed} fold {fold}: accuracy {run['accuracies'][fold]:.4f} | macro F1 {run['f1_scores'][fold]:.4f}")
    pd.DataFrame(rows).to_csv(results_folder / name / "cv_scores.csv", index=False)

def load_scores(kind="cv"):
    """All saved scores of one kind ("cv" or "test"), one row per (model, seed, fold)."""
    frames = [pd.read_csv(path).assign(model=path.parent.name) for path in sorted(results_folder.glob(f"*/{kind}_scores.csv"))]
    return pd.concat(frames, ignore_index=True)[["model", "seed", "fold", "accuracy", "f1"]]

def cv_data_frame(metric="f1"):
    """One row per (model, seed, fold) with the saved cross-validation score ("accuracy" or "f1"), e.g. for the ROPE analysis."""
    return load_scores("cv").rename(columns={metric: "score"})[["model", "seed", "fold", "score"]]

def evaluate_on_test(name, test_pairs, encoder, model_option):
    """Loads every fold model of experiment `name`, predicts the test set and saves accuracy / macro F1 per model
    to results/<name>/test_scores.csv."""
    texts = [text for text, _ in test_pairs]
    gold = [label for _, label in test_pairs]
    rows = []
    for seed, run in zip(my_seeds, my_classifiers[name]):
        for fold, path in run["models"].items():
            if model_option == "logistic":
                preds = part1.loglin.predict(texts, part1.loglin.load(path), encoder)
            else:
                preds = part1.smlp.predict_probs(texts, part1.smlp.load(path), encoder, batch_size=512).argmax(axis=1)
            rows.append({"seed": seed, "fold": fold,
                         "accuracy": accuracy_score(gold, preds),
                         "f1": f1_score(gold, preds, average="macro")})
    scores = pd.DataFrame(rows)
    scores.to_csv(results_folder / name / "test_scores.csv", index=False)
    print(scores.set_index(["seed", "fold"]).round(4).to_string())
    print(f"test accuracy {scores.accuracy.mean():.4f} ± {scores.accuracy.std():.4f} | "
          f"test macro F1 {scores.f1.mean():.4f} ± {scores.f1.std():.4f}")
    return scores

In [25]:
# vocabulary from the train rows only (lemmas)
train_texts = [text for text, _ in make_pairs(train_df, "lemmas", label_names)]
vocab = get_vocab(train_texts, min_count=5, max_size=5000)

In [ ]:
class BagOfWords(Encoder):
    def __init__(self, vocab: dict):
        self.vocab = vocab

    def __len__(self) -> int:
        return len(self.vocab)

    def __call__(self, x: str) -> csr_matrix:
        # repeated words are counted; sparse row so the full dataset fits in memory
        counts = Counter(self.vocab[item] for item in x.split() if item in self.vocab)
        return sparse_row(counts, len(self))

In [27]:
class Embedding(Encoder):
    dense = True  # tells part1 to use dense matrices (and lbfgs for logistic regression) instead of sparse ones

    def __init__(self, embeddings_map: pd.DataFrame):
        # build a sentence -> vector dict once, so each lookup is O(1) instead of scanning the whole dataframe
        self.lookup = {
            sentence: np.asarray(embedding, dtype=np.float32)
            for sentence, embedding in zip(embeddings_map['sentence'], embeddings_map['embeddings'])
        }
    def __len__(self) -> int:
        return 1024
    def __call__(self, x: str) -> np.ndarray:
        return self.lookup[x]

In [28]:
# encoders

token_ind_encoder = TokenIndicator(vocab)

bow_encoder = BagOfWords(vocab)

sentence_em_encoder = Embedding(df)

#print("vocabulary size:", len(token_ind_encoder))

# Experiment 1: Log-Lin

Exp 1.A: Log-Lin with Token Indicator

In [ ]:
my_classifiers["logistic_indicator-lemmas"] = []

for run, seed in enumerate(my_seeds):
    set_seed(seed)
    run_folder = f"{results_folder}/logistic_indicator-lemmas/seed-{seed}"
    my_classifiers["logistic_indicator-lemmas"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=cv_lemmas[run],
            outcome_space=outcome_space,
            encoder=token_ind_encoder,
            model_option="logistic",
            save_to=run_folder,
            # saga needs ~140 iterations to converge here (it stopped at the default of 100)
            loglin_args={
                'max_iter': 1000
            }
        )
    )

accuracies = my_classifiers["logistic_indicator-lemmas"][0]["accuracies"]

for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

In [ ]:
add_cv_f1("logistic_indicator-lemmas", cv_lemmas)
evaluate_on_test("logistic_indicator-lemmas", test_lemmas, token_ind_encoder, "logistic");

Exp 1.B: Log-Lin with Bag of Words

In [ ]:
my_classifiers["logistic_bow-lemmas"] = []

for run, seed in enumerate(my_seeds):
    set_seed(seed)
    run_folder = f"{results_folder}/logistic_bow-lemmas/seed-{seed}"
    my_classifiers["logistic_bow-lemmas"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=cv_lemmas[run],
            outcome_space=outcome_space,
            encoder=bow_encoder,
            model_option="logistic",
            save_to=run_folder,
            # lbfgs: saga had not converged on the raw counts even after 1000 iterations (it needs ~2,200);
            # lbfgs reaches the same solution in ~300 iterations
            loglin_args={
                'solver': 'lbfgs',
                'max_iter': 1000
            }
        )
    )

accuracies = my_classifiers["logistic_bow-lemmas"][0]["accuracies"]

for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

In [ ]:
add_cv_f1("logistic_bow-lemmas", cv_lemmas)
evaluate_on_test("logistic_bow-lemmas", test_lemmas, bow_encoder, "logistic");

Exp 1.C: Log-Lin with Sentence Embeddings

In [ ]:
my_classifiers["logistic_embeddings"] = []

for run, seed in enumerate(my_seeds):
    set_seed(seed)
    run_folder = f"{results_folder}/logistic_embeddings/seed-{seed}"
    my_classifiers["logistic_embeddings"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=cv_sentences[run],
            outcome_space=outcome_space,
            encoder=sentence_em_encoder,
            model_option="logistic",
            save_to=run_folder,
            loglin_args= {
                'max_iter': 300
            }
        )
    )

accuracies = my_classifiers["logistic_embeddings"][0]["accuracies"]

for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

In [ ]:
add_cv_f1("logistic_embeddings", cv_sentences)
evaluate_on_test("logistic_embeddings", test_sentences, sentence_em_encoder, "logistic");

# Experiment 2: Multi-Layer Perceptron
One hidden layer MLP

Exp 2.A: MLP with Token Indicator

In [ ]:
my_classifiers["mlp_indicator-lemmas"] = []

for run, seed in enumerate(my_seeds):
    set_seed(seed)
    run_folder = f"{results_folder}/mlp_indicator-lemmas/seed-{seed}"
    my_classifiers["mlp_indicator-lemmas"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=cv_lemmas[run],
            outcome_space=outcome_space,
            encoder=token_ind_encoder,
            model_option="mlp",
            save_to=run_folder,
            # tuned with mlp_grid_search.py / mlp_grid_check.py (see grid_search/)
            smlp_args={
                "hidden_dim": 64,
                "dropout": 0.3,
                "batch_size": 128,
                "learning_rate": 3e-4,
                "weight_decay": 1e-4,
                "num_epochs": 25,
            },
        )
    )

accuracies = my_classifiers["mlp_indicator-lemmas"][0]["accuracies"]

for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

In [ ]:
add_cv_f1("mlp_indicator-lemmas", cv_lemmas)
evaluate_on_test("mlp_indicator-lemmas", test_lemmas, token_ind_encoder, "mlp");

Exp 2.B: MLP with Bag of Words

In [ ]:
my_classifiers["mlp_bow-lemmas"] = []

for run, seed in enumerate(my_seeds):
    set_seed(seed)
    run_folder = f"{results_folder}/mlp_bow-lemmas/seed-{seed}"
    my_classifiers["mlp_bow-lemmas"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=cv_lemmas[run],
            outcome_space=outcome_space,
            encoder=bow_encoder,
            model_option="mlp",
            save_to=run_folder,
            # tuned with mlp_grid_search.py / mlp_grid_check.py (see grid_search/)
            smlp_args={
                "hidden_dim": 64,
                "dropout": 0.3,
                "batch_size": 128,
                "learning_rate": 1e-3,
                "weight_decay": 1e-4,
                "num_epochs": 25,
            },
        )
    )

accuracies = my_classifiers["mlp_bow-lemmas"][0]["accuracies"]

for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

In [ ]:
add_cv_f1("mlp_bow-lemmas", cv_lemmas)
evaluate_on_test("mlp_bow-lemmas", test_lemmas, bow_encoder, "mlp");

Exp 2.C: MLP with Sentence Embeddings

In [ ]:
my_classifiers["mlp_embeddings"] = []

for run, seed in enumerate(my_seeds):
    set_seed(seed)
    run_folder = f"{results_folder}/mlp_embeddings/seed-{seed}"
    my_classifiers["mlp_embeddings"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=cv_sentences[run],
            outcome_space=outcome_space,
            encoder=sentence_em_encoder,
            model_option="mlp",
            save_to=run_folder,
            # tuned with mlp_grid_search.py / mlp_grid_check.py (see grid_search/): this setting was already the best
            smlp_args={
                "hidden_dim": 128,
                "dropout": 0.1,
                "batch_size": 512,
                "learning_rate": 1e-3,
                "weight_decay": 1e-4,
                "num_epochs": 25,
            },
        )
    )

accuracies = my_classifiers["mlp_embeddings"][0]["accuracies"]

for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

In [ ]:
add_cv_f1("mlp_embeddings", cv_sentences)
evaluate_on_test("mlp_embeddings", test_sentences, sentence_em_encoder, "mlp");

# Experiment 3: Two-Hidden-Layer MLP
Backup alternative method: the Experiment 2 MLP with a second hidden layer of the same size.
Same hyperparameters as the matching Experiment 2 run (not tuned separately), so the only change is the extra layer.

Exp 3.A: Two-Layer MLP with Token Indicator

In [ ]:
my_classifiers["mlp2_indicator-lemmas"] = []

for run, seed in enumerate(my_seeds):
    set_seed(seed)
    run_folder = f"{results_folder}/mlp2_indicator-lemmas/seed-{seed}"
    my_classifiers["mlp2_indicator-lemmas"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=cv_lemmas[run],
            outcome_space=outcome_space,
            encoder=token_ind_encoder,
            model_option="mlp",
            save_to=run_folder,
            # Exp 2 settings, with two hidden layers of hidden_dim units
            smlp_args={
                "hidden_dim": 64,
                "num_hidden_layers": 2,
                "dropout": 0.3,
                "batch_size": 128,
                "learning_rate": 3e-4,
                "weight_decay": 1e-4,
                "num_epochs": 25,
            },
        )
    )

accuracies = my_classifiers["mlp2_indicator-lemmas"][0]["accuracies"]

for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

In [ ]:
add_cv_f1("mlp2_indicator-lemmas", cv_lemmas)
evaluate_on_test("mlp2_indicator-lemmas", test_lemmas, token_ind_encoder, "mlp");

Exp 3.B: Two-Layer MLP with Bag of Words

In [ ]:
my_classifiers["mlp2_bow-lemmas"] = []

for run, seed in enumerate(my_seeds):
    set_seed(seed)
    run_folder = f"{results_folder}/mlp2_bow-lemmas/seed-{seed}"
    my_classifiers["mlp2_bow-lemmas"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=cv_lemmas[run],
            outcome_space=outcome_space,
            encoder=bow_encoder,
            model_option="mlp",
            save_to=run_folder,
            # Exp 2 settings, with two hidden layers of hidden_dim units
            smlp_args={
                "hidden_dim": 64,
                "num_hidden_layers": 2,
                "dropout": 0.3,
                "batch_size": 128,
                "learning_rate": 1e-3,
                "weight_decay": 1e-4,
                "num_epochs": 25,
            },
        )
    )

accuracies = my_classifiers["mlp2_bow-lemmas"][0]["accuracies"]

for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

In [ ]:
add_cv_f1("mlp2_bow-lemmas", cv_lemmas)
evaluate_on_test("mlp2_bow-lemmas", test_lemmas, bow_encoder, "mlp");

Exp 3.C: Two-Layer MLP with Sentence Embeddings

In [ ]:
my_classifiers["mlp2_embeddings"] = []

for run, seed in enumerate(my_seeds):
    set_seed(seed)
    run_folder = f"{results_folder}/mlp2_embeddings/seed-{seed}"
    my_classifiers["mlp2_embeddings"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=cv_sentences[run],
            outcome_space=outcome_space,
            encoder=sentence_em_encoder,
            model_option="mlp",
            save_to=run_folder,
            # Exp 2 settings, with two hidden layers of hidden_dim units
            smlp_args={
                "hidden_dim": 128,
                "num_hidden_layers": 2,
                "dropout": 0.1,
                "batch_size": 512,
                "learning_rate": 1e-3,
                "weight_decay": 1e-4,
                "num_epochs": 25,
            },
        )
    )

accuracies = my_classifiers["mlp2_embeddings"][0]["accuracies"]

for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

In [ ]:
add_cv_f1("mlp2_embeddings", cv_sentences)
evaluate_on_test("mlp2_embeddings", test_sentences, sentence_em_encoder, "mlp");

In [ ]:
# summary: cross-validation and test accuracy / macro F1 for every experiment (mean and std over the fold models),
# read from the score files in results/, so this also works after a kernel restart
summary = pd.concat({
    "cv": load_scores("cv").groupby("model")[["accuracy", "f1"]].agg(["mean", "std"]),
    "test": load_scores("test").groupby("model")[["accuracy", "f1"]].agg(["mean", "std"]),
}, axis=1)
majority = test_df[LABEL_COL].value_counts(normalize=True).max()
print(f"majority-class baseline: accuracy {majority:.4f}, macro F1 {majority / (1 + majority):.4f}")
summary.round(4)